In [ ]:
# jupyter notebook 전체화면으로 변경  
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [1]:
# load module
import os
#os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import pytorch_lightning as pl
import seaborn as sns

from my_funs import *
from plot_round_cut import *


from itertools import product
from sklearn import metrics
from matplotlib import gridspec
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger


from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

#import tensorflow as tf 
import tensorboard as tb 
#tf.io.gfile = tb.compat.tensorflow_stub.io.gfile

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'

dongs = ['강남동', '교  동', '근화동', '남  면', '남산면', '동  면', '동내면', '동산면', '북산면',
       '사북면', '서  면', '석사동', '소양동', '신동면', '신북읍', '신사우동', '약사명동', '조운동',
       '퇴계동', '효자1동', '후평1동']

/home/nplab/.local/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /home/nplab/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will n

In [2]:
# epoch 100
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[0]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
13,2,0.5,0.121193
4,1,0.5,0.121193
22,3,0.5,0.121193
3,1,0.4,0.118180
21,3,0.4,0.118180
12,2,0.4,0.118180
2,1,0.3,0.113212
20,3,0.3,0.113212
11,2,0.3,0.113212
10,2,0.2,0.104325


In [3]:
# epoch 200
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[1]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 200 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.133003
20,3,0.3,0.133003
11,2,0.3,0.133003
3,1,0.4,0.127762
21,3,0.4,0.127762
12,2,0.4,0.127762
13,2,0.5,0.124138
4,1,0.5,0.124138
22,3,0.5,0.124138
10,2,0.2,0.118192


In [4]:
# epoch 300
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[2]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 300 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.107382
20,3,0.3,0.107382
11,2,0.3,0.107382
1,1,0.2,0.103014
19,3,0.2,0.103014
10,2,0.2,0.103014
3,1,0.4,0.101273
21,3,0.4,0.101273
12,2,0.4,0.101273
13,2,0.5,0.088844


In [5]:
# epoch 300
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[2]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 300 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.107382
20,3,0.3,0.107382
11,2,0.3,0.107382
1,1,0.2,0.103014
19,3,0.2,0.103014
10,2,0.2,0.103014
3,1,0.4,0.101273
21,3,0.4,0.101273
12,2,0.4,0.101273
13,2,0.5,0.088844


In [6]:
# epoch 400
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[3]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 400 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.118736
20,3,0.3,0.118736
11,2,0.3,0.118736
1,1,0.2,0.112749
19,3,0.2,0.112749
10,2,0.2,0.112749
3,1,0.4,0.101541
21,3,0.4,0.101541
12,2,0.4,0.101541
0,1,0.1,0.094050


In [2]:
# epoch 500
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[4]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 500 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
3,1,0.4,0.125472
21,3,0.4,0.125472
12,2,0.4,0.125472
13,2,0.5,0.119812
4,1,0.5,0.119812
22,3,0.5,0.119812
2,1,0.3,0.117579
20,3,0.3,0.117579
11,2,0.3,0.117579
10,2,0.2,0.103102


In [3]:
# epoch 600
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[5]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 600 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.110319
20,3,0.3,0.110319
11,2,0.3,0.110319
3,1,0.4,0.109007
21,3,0.4,0.109007
12,2,0.4,0.109007
1,1,0.2,0.106793
19,3,0.2,0.106793
10,2,0.2,0.106793
13,2,0.5,0.102987


In [4]:
# epoch 700
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[6]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 700 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.116363
20,3,0.3,0.116363
11,2,0.3,0.116363
1,1,0.2,0.114992
19,3,0.2,0.114992
10,2,0.2,0.114992
3,1,0.4,0.113450
21,3,0.4,0.113450
12,2,0.4,0.113450
13,2,0.5,0.095163


In [5]:
# epoch 800
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[7]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 800 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
3,1,0.4,0.128848
21,3,0.4,0.128848
12,2,0.4,0.128848
13,2,0.5,0.122147
4,1,0.5,0.122147
22,3,0.5,0.122147
2,1,0.3,0.117972
20,3,0.3,0.117972
11,2,0.3,0.117972
1,1,0.2,0.107048


In [6]:
# epoch 900
setting = pd.read_csv('Atten_head_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[8]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 900 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
3,1,0.4,0.124418
21,3,0.4,0.124418
12,2,0.4,0.124418
2,1,0.3,0.114335
20,3,0.3,0.114335
11,2,0.3,0.114335
13,2,0.5,0.112434
4,1,0.5,0.112434
22,3,0.5,0.112434
10,2,0.2,0.104399
